# Imports

In [2]:
import sys
sys.path.append("..")

In [3]:
import os
import re
import logging
import json
from datasets import load_dataset, Dataset

In [4]:
def log_module(name, level):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    ch = logging.StreamHandler()
    ch.setLevel(level)

    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)

    logger.addHandler(ch)

In [5]:
log_module("GraphEvolve", logging.DEBUG)
log_module("AGoTI", logging.WARNING)

In [6]:
from GraphEvolve.task import SimpleTask
from GraphEvolve.goo import get_default_config
from GraphEvolve.planner import ThreeStepPlanner
from GraphEvolve.solution_database import SolutionDatabase
from GraphEvolve.sampler import PowerLawSampler
from GraphEvolve.mutator import EditMutator, PlanMutator, RandomMutator
from GraphEvolve.runner import Runner

In [7]:
from AGoTI.model import ApiVLLMModel

# Models

In [8]:
generation_config = {
    "temperature": 0.6,
    "top_p": 0.9,
    "max_tokens": 5000,
    "extra_body": {
        "repetition_penalty": 1.0,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None
        }
}

In [9]:
runner_generation_config = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 10,
    "frequency_penalty": 0.1,
    "extra_body": {
        "repetition_penalty": 1.0,
        "min_p": 0.8,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None,
        "chat_template_kwargs": {
            "enable_thinking": False
        },
    }
}

In [ ]:
API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
model_name = "Qwen3-235B-A22B-Instruct-2507"
runner_model_name = "tpro-2"
model = ApiVLLMModel(API_KEY, API_URL, model_name, generation_config=generation_config, log_to_file=True)
runner_model = ApiVLLMModel(API_KEY, API_URL, runner_model_name, generation_config=runner_generation_config)

# Task

In [12]:
ALL_TAGS_NEREL = [
    "AGE", "AWARD", "CITY", "COUNTRY", "CRIME", "DATE", "DISEASE", "DISTRICT",
    "EVENT", "FACILITY", "FAMILY", "IDEOLOGY", "LANGUAGE", "LAW", "LOCATION",
    "MONEY", "NATIONALITY", "NUMBER", "ORDINAL", "ORGANIZATION", "PENALTY",
    "PERCENT", "PERSON", "PRODUCT", "PROFESSION", "RELIGION", "STATE_OR_PROVINCE",
    "TIME", "WORK_OF_ART"
]

# REMOVE_TAGS = ["EVENT", "LAW"]
REMOVE_TAGS = []
DEFAULT_TAGS_NEREL = [item for item in ALL_TAGS_NEREL if item not in REMOVE_TAGS]

TASK_DESCRIPTION = '''Твоя задача — найти ВСЕ именованные сущности в тексте, включая ВЛОЖЕННЫЕ (когда одна сущность полностью содержится внутри другой). Если одна и та же сущность встречается несколько раз, извлеки каждое вхождение в порядке их появления в тексте.

Категории сущностей (теги):

*ЛЮДИ И РОЛИ*
- PERSON: Имена и фамилии людей (включая инициалы, сокращённые формы, обращения: "Сешнс", "Рахимов", "Дэвид Саутеру").
- PROFESSION: Должности, звания, титулы (включая составные: "президент Башкирии", "глава пресс-службы", "гендиректор ОАО").
- FAMILY: Фамилии как обозначение династии или семьи (если явно указано).
- NATIONALITY: Национальности, этнонимы, религиозные или политические группы (например: "русская", "мусульмане", "демократы", "республиканцы").

*ОРГАНИЗАЦИИ И МЕСТА*
- ORGANIZATION: Организации, компании, партии, СМИ, комитеты, учреждения (например: "Единая Россия", "Los Angeles Times", "юридический комитет").
- FACILITY: Здания, сооружения, инфраструктура (стадионы, мосты, музеи как объекты).
- COUNTRY: Названия стран и прилагательные, относящиеся к государствам (например: "США", "российская власть").
- CITY: Названия городов и прилагательные от них (например: "Уфы", "московский").
- STATE_OR_PROVINCE: Регионы, штаты, области, республики (например: "Алабама", "Башкирия").
- DISTRICT: Районы, округи внутри городов или регионов.
- LOCATION: Природные и географические объекты (горы, реки, континенты), а также абстрактные места (например: "федеральный центр").

*СОБЫТИЯ И ЗАКОН*
- EVENT: События, процессы, действия (например: "отставка", "выборы", "подписал указ", "назначение").
- CRIME: Преступления (вымогательство, коррупция).
- LAW: Названия законов, кодексов.
- PENALTY: Виды наказаний.
- DISEASE: Болезни, вирусы.

*ЧИСЛА И ВРЕМЯ*
- DATE: Даты (полные или частичные: "вчера", "6 июня", "в июне", "с 1997 года").
- TIME: Время суток.
- AGE: Возраст (например: "42-летний").
- NUMBER: Числовые значения (не даты и не деньги).
- ORDINAL: Порядковые числительные ("первый", "2-й").
- PERCENT: Проценты.
- MONEY: Денежные суммы.

*ПРОЧЕЕ*
- WORK_OF_ART: Книги, фильмы, газеты, журналы, СМИ (например: "Московский комсомолец", "Ъ").
- PRODUCT: Товары, техника, оружие.
- AWARD: Награды, премии.
- LANGUAGE: Языки.
- RELIGION: Религии.
- IDEOLOGY: Идеологии, политические направления (например: "консерватор", "демократ", "коммунизм").

ВАЖНЫЕ ПРАВИЛА:
1. Извлекай ВСЕ сущности, включая вложенные. Например, в "Президент Башкирии Муртаза Рахимов" должно быть:
   - ["PROFESSION", "Президент Башкирии"]
   - ["STATE_OR_PROVINCE", "Башкирии"]
   - ["PERSON", "Муртаза Рахимов"]
2. Не изменяй текст сущности — копируй его дословно, включая склонения, регистр и пунктуацию.
3. Сохраняй порядок сущностей строго по их первому появлению в тексте. Если сущность начинается раньше — она должна быть раньше в списке.
4. Не объединяй сущности. Каждая должна быть отдельным элементом.
5. Извлекай даже короткие формы: "Сешнс", "Хабирова", "Ъ", "РФ".
6. Прилагательные, образованные от географических названий, могут быть отдельными сущностями (например: "московский" → CITY, "башкирский" → STATE_OR_PROVINCE).
7. Политические и идеологические термины (республиканец, демократ, консерватор) — это IDEOLOGY, если не указано как должность.
8. СМИ и газеты — WORK_OF_ART, если не указано как организация (но в примерах "Ъ" и "Los Angeles Times" — ORGANIZATION).
9. Действия и процессы (уволил, отставка, назначение, покинуть) — это EVENT.
10. Должности, содержащие географические названия, могут включать вложенные сущности (например: "глава башкирской администрации" → PROFESSION + STATE_OR_PROVINCE).

Формат вывода:
Только JSON-список списков: [["TAG", "текст сущности"], ...]
Без пояснений, без комментариев, без дополнительного текста.

Текст для обработки обозначается строкой {text}.
Ответ должен быть в пункте \"entities\"'''


In [13]:
def process_entities(sample):
    entities = []
    for entity in sample["entities"]:
        match = re.match(r"(?P<id>\w+)\s(?P<tag>\w+)\s(?P<b_e_pairs>\d+\s\d+(?:;\d+\s\d+)*)\s(?P<text>.+)", entity)
        tag = match.group("tag")
        b_e_pairs = match.group("b_e_pairs").split(';')
        text = match.group("text")
        for b_e_pair in b_e_pairs:
            begin, end = map(int, b_e_pair.split())
            entities.append(
                {
                    "tag": tag,
                    "begin": begin,
                    "end": end,
                    "text": text
                }
            )
    entities.sort(key=lambda x: (x["begin"], -x["end"]))
    return entities

In [14]:
class Multiset():
    def __init__(self, l=None):
        if l is None:
            l = []
        if type(l) == list:
            data = {}
            for e in l:
                if e in data.keys():
                    data[e] += 1
                else:
                    data[e] = 1
            self.data = data
        elif type(l) == dict:
            self.data = l
        else:
            raise Exception("Multiset can be initialized only with list or dictionary")

    def count(self):
        count = 0
        for v in self.data.values():
            count += v
        return count
    
    def union(self, m):
        data = self.data.copy()
        for k, v in m.data.items():
            if k in self.data.keys():
                data[k] = max(data[k], v)
            else:
                data[k] = v
        return Multiset(data)

    def intersect(self, m):
        data = {}
        for k, v in self.data.items():
            if k in m.data.keys():
                data[k] = min(v, m.data[k])
        return Multiset(data)

    def subtract(self, m):
        data = {}
        for k, v in self.data.items():
            if k in m.data.keys():
                if self.data[k] > m.data[k]:
                    data[k] = v - m.data[k]
            else:
                data[k] = v
        return Multiset(data)

    def add(self, m):
        data = self.data.copy()
        for k, v in m.data.items():
            if k in self.data.keys():
                data[k] += v
            else:
                data[k] = v
        return Multiset(data)

In [15]:
def list_to_dict_multiset(l, tags):
    d = {tag: [] for tag in tags}
    tags = set(tags)
    for item in l:
        if not isinstance(item, list) or len(item) != 2:
            continue
        tag, entity = item
        if isinstance(tag, str) and tag in tags:
            d[tag].append(entity)
    for tag in tags:
        d[tag] = Multiset(list(filter(lambda x: isinstance(x, str), d[tag])))
    return d

In [16]:
def extract_answer(gen_pred):
    try:
        gen_pred = gen_pred.replace('```json', '').strip()
        gen_pred = gen_pred.replace('```', '').strip()
        predict = json.loads(gen_pred)
    except:
        predict = []
    return predict

In [17]:
def get_answer(sample):
    tagged_tokens = []
    for entity in sample["entities"]:
        if entity["tag"] in ALL_TAGS_NEREL:
            tagged_tokens.append([entity["tag"], entity["text"]])
    return tagged_tokens

In [18]:
def get_tp_fn_fp(sample, gen_pred, return_dict=False):
    y_pred = extract_answer(gen_pred)
    y_gold = get_answer(sample)

    pred_tags = set(map(lambda x: x[0], y_pred))
    gold_tags = set(map(lambda x: x[0], y_gold))
    combined_tags = set(list(pred_tags) + list(gold_tags))
    
    y_pred = list_to_dict_multiset(y_pred, combined_tags)
    y_gold = list_to_dict_multiset(y_gold, combined_tags)

    true_positives = {tag: y_pred[tag].intersect(y_gold[tag]) for tag in combined_tags}
    false_negatives = {tag: y_gold[tag].subtract(y_pred[tag]) for tag in combined_tags}
    false_positives = {tag: y_pred[tag].subtract(y_gold[tag]) for tag in combined_tags}

    true_positives = {k: v for k, v in true_positives.items() if v.data != {}}
    false_negatives = {k: v for k, v in false_negatives.items() if v.data != {}}
    false_positives = {k: v for k, v in false_positives.items() if v.data != {}}

    true_positives_count = {k: v.count() for k, v in true_positives.items()}
    false_negatives_count = {k: v.count() for k, v in false_negatives.items()}
    false_positives_count = {k: v.count() for k, v in false_positives.items()}

    if return_dict:
        return (true_positives, false_negatives, false_positives)
    else:
        return (true_positives_count, false_negatives_count, false_positives_count)

In [19]:
def f1(tp_fn_fp_counts):
    tt_count, fn_count, fp_count = tp_fn_fp_counts
    tt_count = sum(list(tt_count.values()))
    fn_count = sum(list(fn_count.values()))
    fp_count = sum(list(fp_count.values()))
    score = (2 * tt_count) / (2 * tt_count + fn_count + fp_count)
    return score

In [20]:
def f1_macro(tp_fn_fp):
    tp_total = {k: 0 for k in ALL_TAGS_NEREL}
    fn_total = {k: 0 for k in ALL_TAGS_NEREL}
    fp_total = {k: 0 for k in ALL_TAGS_NEREL}

    for s in tp_fn_fp:
        tt, fn, fp = s
        for k in tt.keys():
            if k in ALL_TAGS_NEREL:
                tp_total[k] += tt[k]
        for k in fn.keys():
            if k in ALL_TAGS_NEREL:
                fn_total[k] += fn[k]
        for k in fp.keys():
            if k in ALL_TAGS_NEREL:
                fp_total[k] += fp[k]

    f1_score = [2 * tp_total[tag] / (2 * tp_total[tag] + fp_total[tag] + fn_total[tag]) \
          if tp_total[tag] > 0 else 0 for tag in ALL_TAGS_NEREL if tp_total[tag] + fp_total[tag] + fn_total[tag] > 0]
    return sum(f1_score) / len(f1_score)

In [21]:
def get_answer_str(sample):
    answer = get_answer(sample)
    answer_str = '```json\n' + json.dumps(answer, ensure_ascii=False, indent=4).strip() + '\n```'
    return answer_str

In [ ]:
class NestedNerJson(SimpleTask):
    def __init__(self):
        self.description = TASK_DESCRIPTION

    def dataset_args(self):
        return {"path": "MalakhovIlya/NEREL", "name": "data", "trust_remote_code": True}

    def load_dataset(self):
        dataset = load_dataset(**self.dataset_args())
        train_dataset = self.process_dataset(dataset[self.train_split()])
        test_dataset = self.process_dataset(dataset[self.test_split()])

        train_dataset = train_dataset.select(range(min(10, len(test_dataset))))
        test_dataset = test_dataset.select(range(min(100, len(test_dataset))))
        return (train_dataset, test_dataset)

    def process_dataset(self, dataset: Dataset):
        return dataset.map(self.process_sample)

    def process_sample(self, sample):
        entities = process_entities(sample)
        return {
            "query": sample["text"],
            "entities": entities
        }

    def train_split(self):
        return "train"

    def test_split(self):
        return "test"

    async def evaluate(self, sample, output):
        y_pred = output.get("entities", [""])
        if len(y_pred) == 0:
            y_pred = [""]
        y_pred = y_pred[0]
        return get_tp_fn_fp(sample, y_pred, return_dict=False)

    async def evaluate_dataset(self, samples, outputs, scores):
        score = await self.aggregate(scores)
        scores = [f1(tp_fn_fp) for tp_fn_fp in scores]
        return score, scores

    async def aggregate(self, scores):
        return f1_macro(scores)

    async def generate_feedback(self, sample, output) -> str:
        _, fn, fp = get_tp_fn_fp(sample, output, return_dict=True)

        feedback = ""
        if fn != {}:
            feedback += "Following entities were missed:\n"
            for tag, entities in fn.items():
                for entity, count in entities.data.items():
                    feedback += f"- \"{entity}\" (tag: {tag})" + (f" ({count} times)" if count > 1 else "") + "\n"
        if fp != {}:
            feedback += "\nFollowing entities were extracted, but shouldn't have been:\n"
            for tag, entities in fp.items():
                for entity, count in entities.data.items():
                    feedback += f"- \"{entity}\" (tag: {tag})" + (f" ({count} times)" if count > 1 else "") + "\n"
        if fn == {} and fp == {}:
            feedback = "Everything is correct!"
        
        return feedback

    def get_goo_kwargs(self, sample):
        return {"replace": {"{text}": sample["query"]}}

In [23]:
task = NestedNerJson()

# GoO config

In [24]:
goo_config = get_default_config(runner_model)

# Planner

In [25]:
planner = ThreeStepPlanner(model, goo_config)

# Solution database

In [ ]:
solution_db = SolutionDatabase(goo_config, clear_db=False, db_path="./nerel_solution_database.db")

# Sampler

In [26]:
sampler = PowerLawSampler(solution_db)

# Mutator

In [27]:
edit_mutator = EditMutator(model, goo_config, worst_n=2, max_changes=2)
plan_mutator = PlanMutator(model, goo_config, planner, worst_n=1)
mutator = RandomMutator([edit_mutator, plan_mutator], [0.75, 0.25], ["edit", "plan"])

# Basic solution

In [28]:
basic_plan_draft = """1. Извлекает из текста (обозначается строкой {text}) именнованные сущности (включая вложенные)."""

In [29]:
basic_plan_formal = """\
1. "корень"
"Извлечение именованных сущностей"
"Извлекает из текста (обозначается строкой {text}) именнованные сущности (включая вложенные) следующих классов: AGE, AWARD, CITY, COUNTRY, CRIME, DATE, DISEASE, DISTRICT, EVENT, FACILITY, FAMILY, IDEOLOGY, LANGUAGE, LAW, LOCATION, MONEY, NATIONALITY, NUMBER, ORDINAL, ORGANIZATION, PENALTY, PERCENT, PERSON, PRODUCT, PROFESSION, RELIGION, STATE_OR_PROVINCE, TIME, WORK_OF_ART. Результат — JSON-список списков: [["TAG", "текст сущности"], ...]"
входы: []
выход: entities

Ответы: {"entities": 1}"""

In [30]:
NEREL_JSON_INSTRUCTION = '''Ты — эксперт по извлечению именованных сущностей из русскоязычных текстов. Твоя задача — найти ВСЕ именованные сущности в тексте, включая ВЛОЖЕННЫЕ (когда одна сущность полностью содержится внутри другой). Если одна и та же сущность встречается несколько раз, извлеки каждое вхождение в порядке их появления в тексте.

Категории сущностей (теги):

*ЛЮДИ И РОЛИ*
- PERSON: Имена и фамилии людей (включая инициалы, сокращённые формы, обращения: "Сешнс", "Рахимов", "Дэвид Саутеру").
- PROFESSION: Должности, звания, титулы (включая составные: "президент Башкирии", "глава пресс-службы", "гендиректор ОАО").
- FAMILY: Фамилии как обозначение династии или семьи (если явно указано).
- NATIONALITY: Национальности, этнонимы, религиозные или политические группы (например: "русская", "мусульмане", "демократы", "республиканцы").

*ОРГАНИЗАЦИИ И МЕСТА*
- ORGANIZATION: Организации, компании, партии, СМИ, комитеты, учреждения (например: "Единая Россия", "Los Angeles Times", "юридический комитет").
- FACILITY: Здания, сооружения, инфраструктура (стадионы, мосты, музеи как объекты).
- COUNTRY: Названия стран и прилагательные, относящиеся к государствам (например: "США", "российская власть").
- CITY: Названия городов и прилагательные от них (например: "Уфы", "московский").
- STATE_OR_PROVINCE: Регионы, штаты, области, республики (например: "Алабама", "Башкирия").
- DISTRICT: Районы, округи внутри городов или регионов.
- LOCATION: Природные и географические объекты (горы, реки, континенты), а также абстрактные места (например: "федеральный центр").

*СОБЫТИЯ И ЗАКОН*
- EVENT: События, процессы, действия (например: "отставка", "выборы", "подписал указ", "назначение").
- CRIME: Преступления (вымогательство, коррупция).
- LAW: Названия законов, кодексов.
- PENALTY: Виды наказаний.
- DISEASE: Болезни, вирусы.

*ЧИСЛА И ВРЕМЯ*
- DATE: Даты (полные или частичные: "вчера", "6 июня", "в июне", "с 1997 года").
- TIME: Время суток.
- AGE: Возраст (например: "42-летний").
- NUMBER: Числовые значения (не даты и не деньги).
- ORDINAL: Порядковые числительные ("первый", "2-й").
- PERCENT: Проценты.
- MONEY: Денежные суммы.

*ПРОЧЕЕ*
- WORK_OF_ART: Книги, фильмы, газеты, журналы, СМИ (например: "Московский комсомолец", "Ъ").
- PRODUCT: Товары, техника, оружие.
- AWARD: Награды, премии.
- LANGUAGE: Языки.
- RELIGION: Религии.
- IDEOLOGY: Идеологии, политические направления (например: "консерватор", "демократ", "коммунизм").

ВАЖНЫЕ ПРАВИЛА:
1. Извлекай ВСЕ сущности, включая вложенные. Например, в "Президент Башкирии Муртаза Рахимов" должно быть:
   - ["PROFESSION", "Президент Башкирии"]
   - ["STATE_OR_PROVINCE", "Башкирии"]
   - ["PERSON", "Муртаза Рахимов"]
2. Не изменяй текст сущности — копируй его дословно, включая склонения, регистр и пунктуацию.
3. Сохраняй порядок сущностей строго по их первому появлению в тексте. Если сущность начинается раньше — она должна быть раньше в списке.
4. Не объединяй сущности. Каждая должна быть отдельным элементом.
5. Извлекай даже короткие формы: "Сешнс", "Хабирова", "Ъ", "РФ".
6. Прилагательные, образованные от географических названий, могут быть отдельными сущностями (например: "московский" → CITY, "башкирский" → STATE_OR_PROVINCE).
7. Политические и идеологические термины (республиканец, демократ, консерватор) — это IDEOLOGY, если не указано как должность.
8. СМИ и газеты — WORK_OF_ART, если не указано как организация (но в примерах "Ъ" и "Los Angeles Times" — ORGANIZATION).
9. Действия и процессы (уволил, отставка, назначение, покинуть) — это EVENT.
10. Должности, содержащие географические названия, могут включать вложенные сущности (например: "глава башкирской администрации" → PROFESSION + STATE_OR_PROVINCE).

Формат вывода:
Только JSON-список списков: [["TAG", "текст сущности"], ...]
Без пояснений, без комментариев, без дополнительного текста.

Текст для обработки:
{text}'''

basic_goo_json = {'nodes': {'1': {'class': 'корень',
   'parents': [],
   'args': {'prompt': NEREL_JSON_INSTRUCTION,
    'parsing': 'none',
    'thought_tag': 'entities',
    'name': "Извлечение именованных сущностей",
    'description': "Извлекает из текста (обозначается строкой {text}) именнованные сущности (включая вложенные) следующих классов: " \
                   "AGE, AWARD, CITY, COUNTRY, CRIME, DATE, DISEASE, DISTRICT, EVENT, FACILITY, FAMILY, IDEOLOGY, " \
                   "LANGUAGE, LAW, LOCATION, MONEY, NATIONALITY, NUMBER, ORDINAL, ORGANIZATION, PENALTY, PERCENT, " \
                   "PERSON, PRODUCT, PROFESSION, RELIGION, STATE_OR_PROVINCE, TIME, WORK_OF_ART. " \
                   "Результат — JSON-список списков: [[\"TAG\", \"текст сущности\"], ...]"
   }}},
"roots": [1],
"outputs": {"entities": 1}}

# Runner

In [ ]:
runner = Runner(task, solution_db, goo_config, planner, sampler, mutator)

In [ ]:
INIT_WITH_BASIC_SOLUTION = True

In [ ]:
if INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution(basic_plan_draft, basic_plan_formal, basic_goo_json, None)
    solution_db.add_root(basic_solution)

In [ ]:
await runner.run(1.0, 10)

# Results

In [ ]:
print("\n".join(f"{score:.3f} (id: {id})" for (score, id), _ in solution_db.solutions._data))

## Initial solution

In [ ]:
print(f"Train: {solution_db.root_solution.score}")
root_score = await runner.run_test(solution_db.root_solution)
print(f"Test: {root_score}")

## Best solution

In [ ]:
print(f"Train: {solution_db.best_solution.score}")
best_score = await runner.run_test(solution_db.best_solution)
print(f"Test: {best_score}")

## Basic solution

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution("", "", basic_goo_json)
    print(f"Train: {basic_solution.score}")

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_score = await runner.run_test(basic_solution)
    print(f"Test: {basic_score}")